# 第 1 课：MDP、Bellman 方程与动态规划

**学习目标**：分清状态、动作、转移概率、奖励、折扣；能够通过迭代得到最优策略。

有限 MDP 用五元组 $(\mathcal S,\mathcal A,P,R,\gamma)$ 描述。$P(s'|s,a)$ 是状态转移概率，$R(s,a,s')$ 是转移奖励。策略 $\pi(a|s)$ 将状态映射到动作分布。

折扣回报定义为

$$ G_t=\sum_{k=0}^{\infty}\gamma^k r_{t+k},\qquad 0\leq\gamma<1. $$

最优状态价值的 Bellman 最优方程：

$$V^*(s)=\max_a\sum_{s'}P(s'|s,a)\left[R(s,a,s')+\gamma V^*(s')\right].$$

等号右侧含有未知的 $V^*$，可以通过反复迭代逼近。

## 1.1 构造一个有终点的 1D 世界

共有 5 个格子：`0 - 1 - 2 - 3 - [4终点]`。动作 `0=左`、`1=右`。向终点前进有每步 -0.02 的代价，进入终点获得 +1。终点是吸收态。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

S, A, gamma = 5, 2, 0.95
P = np.zeros((S, A, S))
R = np.zeros((S, A, S))
for s in range(S):
    for a in range(A):
        ns = s if s == S-1 else int(np.clip(s + (1 if a == 1 else -1), 0, S-1))
        P[s, a, ns] = 1.0
        if s != S-1:
            R[s, a, ns] = 1.0 if ns == S-1 else -0.02
assert np.allclose(P.sum(axis=-1), 1.0)
print("P shape:", P.shape)

## 1.2 把数学写成 NumPy

先计算每个动作的状态-动作价值，再取最大值：

$$Q(s,a)=\sum_{s'}P(s'|s,a)\big[R(s,a,s')+\gamma V(s')\big].$$

`P` 和 `R` 的三根轴分别对应 `(state, action, next_state)`。

In [ ]:
V = np.zeros(S)
errors = []
for it in range(500):
    Q = np.sum(P * (R + gamma * V[None, None, :]), axis=2)
    new_V = np.max(Q, axis=1)
    err = np.max(np.abs(new_V - V))
    errors.append(err)
    V = new_V
    if err < 1e-10:
        break
policy = np.argmax(Q, axis=1)
print("optimal V:", np.round(V, 3))
print("optimal actions (0=left, 1=right):", policy)
assert np.array_equal(policy[:4], np.ones(4, dtype=int))
plt.figure(figsize=(6,3))
plt.plot(errors)
plt.yscale("log")
plt.xlabel("Value iteration")
plt.ylabel("max Bellman change")
plt.title("Convergence")
plt.show()

## 1.3 解释结果

越接近终点，预期回报越大；折扣 $\gamma$ 越小，远期奖励越不值钱。这里所有状态都能一步步走向终点，所以最优动作为右。

动态规划**已知**环境转移概率；实际 CFD 流动控制一般不知道状态空间的显式转移矩阵，因此接下来学习不要求已知 $P$ 的样本式方法。

### 练习

1. 给“向右”增加 0.1 概率的打滑，写出新的 `P`（需要确保每行概率和为 1）。
2. 修改 `gamma` 为 0.5 和 0.99，对比价值曲线。
3. 增加“动作能耗”项 $-\lambda a^2$，解释为什么这更像流动控制。